# M41 · Relative relocation: differential times, cross-correlation and the double-difference method

**Part 2 · week 5 · about 10 h core**

## 한국어 요약

이 모듈에서는 가까이 모여 있는 지진들의 위치를 서로에 대해 정밀하게 다시 결정하는 이중차분(double-difference) 상대 재배치를 다룹니다. 먼저 두 지진이 같은 관측소에서 보인 주시의 차이(카탈로그 차분 주시)와, 두 파형의 상호상관으로 표본 간격보다 세밀하게 구한 지연 시간을 계산하는 함수를 정답을 아는 합성 자료로 작성하고 점검합니다. 이어서 균질 매질에서 이중차분 방정식 G m = d를 세우고, 감쇠와 평균 이동 구속 조건을 넣어 풀고, 반복 계산으로 수렴시키면서 상대 위치와 클러스터 전체의 절대 위치가 각각 얼마나 정해지는지 확인합니다. 그다음 PocketQuake로 2018-2022년 상주 지진 여섯 개에 HypoDD(ph2dt, 카탈로그 차분 주시, 상호상관 차분 주시)를 실행하고, 여러분의 함수로 얻은 차분 주시와 재배치 결과를 HypoDD의 결과 및 M32의 HYPOINVERSE 위치와 비교합니다. 마지막으로 아무것도 바꾸지 않은 재실행과 시드만 바꾼 부트스트랩으로 결과의 재현성과 불확실성을 살펴보고, 동료가 읽은 결과 파일이 어느 실행에서 나왔는지 실행 기록으로 확인합니다.

## What you will be able to do

- Build catalog differential times for event pairs at common stations, and reproduce HypoDD's dt.ct file from the same phases
- Measure the lag between two waveforms by cross-correlation to a fraction of a sample, test it on made-up shifts, and compare your differential times on the Sangju records with PocketQuake's
- Set up the double-difference system G m = d for a cluster, solve it with damping and a zero mean shift, iterate to convergence, and explain which changes of the event positions and origin times the differential times can and cannot determine
- Relocate the Sangju earthquakes with your own functions and with HypoDD through PocketQuake, and compare both with your HYPOINVERSE locations
- Rerun HypoDD unchanged, bootstrap its uncertainties with two seeds, and show from the run log which run wrote a result file

## Before you start

- Modules: M21, M30, M31, M32
- Reading: Waldhauser & Ellsworth (2000), sections 1 and 2 (the double-difference method and its equations); Waldhauser (2001), the sections on ph2dt and on the input file of hypoDD (weighting, damping, SVD and LSQR); Schaff et al. (2004), the sections on time-domain cross-correlation and on measuring delays below one sample
- Needs: synthetic, stp, pocketquake
- Key terms (see the glossary): absolute location (절대 진원 결정), relative relocation (상대 진원 재결정), station delay (station correction; 관측소 보정값), differential time (도달 시간 차), double difference (이중차분), HypoDD (이중차분 진원 재결정 프로그램), event pair (사건 쌍), travel time (주시), origin time (진원시, 발생 시각), phase pick (위상 피킹, 도달 시각 읽기), ph2dt (ph2dt (차분 주시 생성 프로그램)), outlier (이상값), cross-correlation (상호상관, 교차상관), correlation coefficient (상관 계수), cross-correlation differential time (상호상관 차분 주시), lag (지연 시간(시간 이동)), sub-sample precision (표본 간격보다 세밀한 정밀도), parabolic interpolation (포물선 보간), cycle skipping (주기 건너뜀, 사이클 스키핑), wavelet (웨이블릿(짧은 파형)), zero-phase filter (영위상 필터), sampling rate (표본화율, 샘플링 레이트), partial derivative (편미분), straight ray (직선 파선), Jacobian matrix (야코비 행렬), half-space (반무한 공간), singular value (특이값), null space (영공간), cluster centroid (클러스터 중심), least squares (최소제곱법), damping (감쇠 정규화, 댐핑), damped least squares (감쇠 최소제곱법), mean-shift constraint (평균 이동 구속 조건), condition number (조건수), singular value decomposition (SVD; 특이값 분해), LSQR (LSQR (반복 최소제곱 해법)), linearization (선형화), iteration (반복 계산), observation weight (관측 가중치), RMS residual (RMS 잔차), first arrival (최초 도달파 (초동)), head wave (선두파), crossover distance (교차 거리), HYPOINVERSE (진원 결정 프로그램), PocketQuake (지진 분석 도구 모음), cluster (PocketQuake; 클러스터 (PocketQuake)), dry run (드라이 런, 시험 실행), SAC header (SAC 헤더), checksum (SHA-256 fingerprint; 체크섬), JSON Lines (JSON 줄 형식), run log (실행 기록), return code (exit status; 종료 코드), modification time (수정 시각), stale file (오래된 결과 파일), deterministic computation (결정론적 계산), bootstrap (부트스트랩), bootstrap replica (부트스트랩 반복 표본), percentile (백분위수), random seed (난수 시드), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M41.yaml](../../../.sgtl/criteria/M41.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M41.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster of modules M21 and M32 (located in M32)
PQ_CORES = 3                              # cores of each PocketQuake run (at most 4 on the shared server)
PQ_MEM_GB = 8.0                           # memory cap of each run in GB (HypoDD needs at least 6)
VP_KM_S, VS_KM_S = 5.98, 3.40             # Part 6: speeds of your straight rays: kim1983's top layer (0-15 km)
MODEL_NAME = "kim1983"                    # Part 6: layered model whose crossover distances limit your stations
CC_BAND_HZ = (5.0, 20.0)                  # Part 6: band-pass of the correlation windows (PocketQuake's default)
CC_WINDOW_S = (0.5, 0.5)                  # Part 6: window before and after each pick (s)
MAX_LAG_S = 0.5                           # Part 6: largest lag searched (s)
DAMP = 0.01                               # Part 6: damping of your own relocation
N_ITER = 8                                # Part 6: rounds of your own relocation
BOOT_N = 200                              # Part 7: replicas of each bootstrap
BOOT_SEED = 0                             # Part 7: seed of the first bootstrap
BOOT_SEED_CHANGED = 1000                  # Part 7: the one setting you change, the seed of the second bootstrap
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M41")

import contextlib
import hashlib
import io
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
DD_DIR = RUN_DIR / "2.HypoDD"                                       # what its relocation stages write
RUNS_LOG = st.data.repo_path("work", ".sgtl", "runs.jsonl")         # sgtl's log of your PocketQuake runs
print("PocketQuake results of your cluster:", RUN_DIR)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M41-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up data whose answers are known from how they
were built (`st.reroll("M41-T1")` draws new ones). Task 6 also reads products of your own PocketQuake runs and the
run log; the checks never print the values they compare with.

The order of the work is the one of every module from here on. Tasks 1-5 write the core functions and test them
on made-up clusters with a known answer. Part 6 runs PocketQuake's relocation stages on the six Sangju earthquakes
and applies your functions to the same data, so that you can compare your numbers with an independent program.
Part 7 (task 6) runs the production tool again: an unchanged rerun first, then one changed setting.

This module needs your STP download `sangju2019` (module M03), the course tools (`bin/sgtl-install tools`), and
your PocketQuake cluster located in module M32 (`PQ_SLUG`). Parts 1-5 need none of them. The longest run (the
cross-correlation stage) takes about 8 minutes on 3 cores; never use more than 4 cores on the shared server.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M41.md`.

## Part 0 · Absolute and relative locations (about 20 min)

An **absolute location** (glossary `absolute_location`; modules M31 and M32) places each earthquake on its own,
from its arrival times and a velocity model. Its errors have two sources: the random errors of the picks, and the
errors of the velocity model along the paths to the stations. A slow patch of rock under a station makes every
wave to that station late; module M32 measured such a **station delay** (glossary `station_delay`).

Now take two earthquakes a few hundred metres apart. Their waves travel to a distant station along almost the
same path, so the model error is almost the same for both, and it cancels in the difference of their travel
times. **Relative relocation** (glossary `relative_relocation`) uses such differences to place nearby earthquakes
relative to each other far more precisely than each one can be placed on its own. The **double-difference
method** (glossary `double_difference`; Waldhauser & Ellsworth 2000) and its program **HypoDD** (glossary
`hypodd`; Waldhauser 2001) are the standard way; GrowClust (Trugman & Shearer 2017) is a newer alternative. The
method has relocated whole regional catalogs (Lin et al. 2007; Waldhauser & Schaff 2008).

The six earthquakes of your download, from the KMA catalog that ships with the course (UTC):

In [ ]:
SAC_ROOT = st.data.root() / DATASET / "stp_download" / "SAC"
kma = st.data.sequence("sangju2019")
folders = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()) if SAC_ROOT.is_dir() else []
six = kma[kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S").isin(folders)].reset_index(drop=True)
print(f"{len(folders)} earthquakes in your download {DATASET!r}; in the KMA catalog subset:")
print(six[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))

## Part 1 · Differential times from a catalog (about 20 min, then task 1)

The **travel time** (glossary `travel_time`) of earthquake i to station k is its arrival time there minus its
**origin time** (glossary `origin_time`): $T_i^k = t_i^k - t_{0,i}$. For an **event pair** (glossary `event_pair`)
i and j picked at the same station with the same phase, the **differential time** (glossary `differential_time`)
is

$$dt_k^{ij} = T_i^k - T_j^k = (t_i^k - t_{0,i}) - (t_j^k - t_{0,j}).$$

When the arrival times are **phase picks** (glossary `phase_pick`) and the origin times come from a catalog, these
are the *catalog* differential times. HypoDD's helper program **ph2dt** (glossary `ph2dt`; Waldhauser 2001) forms
them for pairs of neighboring earthquakes and writes them to a file called `dt.ct`. Among its settings, MINOBS is
the smallest number of common station-phases a pair needs to be kept.

### M41-T1 · Catalog differential times · core · about 60 min

**Goal:** write `catalog_dt(picks, origins, min_obs=1)`: the catalog differential times of every event pair at the
stations and phases picked for both events.

**Given:** Part 1 and pandas (a merge of a table with itself is the shortest way). Not allowed: the course package.

**Contract:**
- Inputs: `picks`, a pandas DataFrame with at least the columns `event_id` (text), `station` (text), `phase`
  (`"P"` or `"S"`) and `time_s` (arrival time in s after one reference moment), one row per pick, every event,
  station and phase at most once, in any order and with any index (do not change it); `origins`, a dict from
  event id to origin time (s after the same moment); `min_obs`, an integer of at least 1.
- Output: a DataFrame with the columns `ev1`, `ev2`, `station`, `phase` and `dt_s`: one row per event pair and
  station-phase picked for both events, with `ev1` the smaller id (compared as text), and
  `dt_s = (t_ev1 - t0_ev1) - (t_ev2 - t0_ev2)`. Keep only pairs with at least `min_obs` such rows. Sort the rows
  by `ev1`, `ev2`, `station` and `phase`, with a fresh index 0, 1, 2, ...
- Example: events `"E1"` (origin 100.00 s) and `"E2"` (origin 400.00 s) with P picks at station `"XX.AB01"` at
  105.20 s and 405.05 s give one row: E1, E2, XX.AB01, P, 0.15 s.

**Done when** `st.check("M41-T1", func=catalog_dt)` passes on 12 made-up pick tables (3-6 events, 3-7 stations,
missing picks, shuffled rows, extra columns, several values of `min_obs`).

**Stuck?** `st.hint("M41-T1", 1)`

In [ ]:
# Before you code: the catalog origin time of one event is 0.3 s too late, so all its travel times come out 0.3 s
# short. What happens to the differential times of its pairs: at one station, at all stations, with which sign?
# And what happens if the clock of one station runs 0.3 s late for every event?
st.predict("M41-T1", """

""")

In [ ]:
def catalog_dt(picks, origins, min_obs=1):
    """Catalog differential times of every event pair at the station-phases picked for both events.

    picks   : DataFrame with event_id, station, phase ("P"/"S") and time_s (s after one reference moment); unchanged
    origins : dict event_id -> origin time (s after the same moment)
    min_obs : keep pairs with at least this many common station-phases
    returns : DataFrame with ev1, ev2 (ev1 < ev2 as text), station, phase, dt_s = (t1 - t0_1) - (t2 - t0_2),
              sorted by ev1, ev2, station and phase, with a fresh index
    """
    # Step 1: a travel time for every pick: its time minus the origin time of its event.
    # Step 2: pair the table with itself on station and phase; keep the pairs whose first id is the smaller one.
    # Step 3: dt_s = travel time of ev1 - travel time of ev2; drop pairs with fewer than min_obs rows; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T1: write your code here")

In [ ]:
st.check("M41-T1", func=catalog_dt)

Your function on a made-up catalog of three events at three stations: as it is, with the origin time of E2 made
0.3 s later, and with the clock of station XX.AB02 running 0.3 s late for every event.

In [ ]:
practice = pd.DataFrame({
    "event_id": ["E1", "E1", "E1", "E2", "E2", "E2", "E3", "E3", "E3"],
    "station": ["XX.AB01", "XX.AB02", "XX.AB02", "XX.AB01", "XX.AB02", "XX.AB02", "XX.AB01", "XX.AB02", "XX.AB03"],
    "phase": ["P", "P", "S", "P", "P", "S", "P", "P", "P"],
    "time_s": [105.20, 107.80, 113.50, 405.05, 407.95, 413.70, 705.40, 707.60, 709.10]})
practice_origins = {"E1": 100.0, "E2": 400.0, "E3": 700.0}
as_given = catalog_dt(practice, practice_origins)
late_e2 = catalog_dt(practice, {**practice_origins, "E2": 400.3})
slow_clock = practice.assign(time_s=practice["time_s"] + 0.3 * (practice["station"] == "XX.AB02"))
compare = as_given.assign(**{"E2 origin 0.3 s late": late_e2["dt_s"],
                             "XX.AB02 clock 0.3 s late": catalog_dt(slow_clock, practice_origins)["dt_s"]})
print(compare.round(3).to_string(index=False))
print("\npairs kept with min_obs = 3:", catalog_dt(practice, practice_origins, 3)[["ev1", "ev2"]]
      .drop_duplicates().to_numpy().tolist())

**Explain (3-5 sentences):** Use the printed table. What did the late origin time of E2 do to the differential
times, and what did the late clock of XX.AB02 do? Which of the two errors cancels in a differential time and which
one is left for the relocation to absorb, and how could an inversion absorb it (look ahead at the four unknowns
per event of Part 3)? Why does `min_obs` matter for a pair with only one common station?

## Part 2 · Cross-correlation: a lag to a fraction of a sample (about 30 min, then task 2)

The picks of small earthquakes are uncertain by tens of milliseconds (modules M20 and M21). When two earthquakes
are close together and break in a similar way, their waveforms at a station look alike, and the delay of one
against the other can be measured far more precisely by **cross-correlation** (glossary `cross_correlation`):
slide one window against the other and, for every shift, add up the products of their samples. Removing each
window's mean first and dividing by the square root of the product of the two windows' energies gives the
**correlation coefficient** (glossary `correlation_coefficient`): 1 when the shapes are the same at that shift,
near 0 for unrelated shapes, -1 for an upside-down copy. (The required module M22 slides a template along a long
record the same way.) The shift with the largest coefficient is the **lag** (glossary `lag`) of the second window
against the first. A differential time measured this way is a **cross-correlation differential time** (glossary
`cc_differential_time`; HypoDD's file `dt.cc`).

With samples every 10 ms, the best whole-sample lag can be off by up to half a sample. A **parabola** through the
largest value and its two neighbors gives a lag between samples (**parabolic interpolation**, glossary
`parabolic_interpolation`): **sub-sample precision** (glossary `sub_sample_precision`; Deichmann &
Garcia-Fernandez 1992; Schaff et al. 2004). For values $c_{-1}$, $c_0$ (the largest) and $c_{+1}$ at three
neighboring lags, the top of the parabola lies $\frac{c_{-1} - c_{+1}}{2\,(c_{-1} - 2 c_0 + c_{+1})}$ samples from
the middle one.

One danger: a wave that rings at one frequency matches itself again one full period later. If noise lifts that
neighboring peak above the true one, the lag jumps by a whole period: **cycle skipping** (glossary
`cycle_skipping`). A note on filters: a **zero-phase filter** (glossary `zero_phase_filter`) spreads energy before
the true onset, so picks on such traces come out early (module M20); but a lag between two records filtered the
same way does not move, because both get the same distortion. A filter that is the same for both windows is safe
for cross-correlation.

### M41-T2 · A lag from cross-correlation, to a fraction of a sample · core · about 120 min

**Goal:** write `xcorr_lag(a, b, dt_s, max_lag_s)`: the normalized cross-correlation of two windows for every whole
lag up to `max_lag_s`, and the lag of its largest value refined by a parabola.

**Given:** Part 2 and numpy (sums, products, slicing; `numpy.fft` is allowed). Not allowed: `numpy.correlate`,
`numpy.convolve` (also their `numpy.ma` twins), `numpy.polyfit`, `numpy.polynomial`, `scipy.signal`,
`scipy.ndimage`, `scipy.interpolate`, `scipy.optimize`, `obspy.signal`, `torch`, `statsmodels` and the course
package (the check compares with ObsPy's correlation).

**Contract:**
- Inputs: `a`, `b`: 1-D numpy arrays of the same length (at least 3), sampled every `dt_s` seconds (do not change
  them); `max_lag_s`: the largest lag searched, in s; K = `max_lag_s / dt_s` rounded to the nearest whole number,
  at least 1 and below the length.
- For every whole lag k from -K to K: $c_k = \sum_i x_i\, y_{i+k} \,/\, \sqrt{\sum x^2 \sum y^2}$, where x and y are
  a and b minus their means and the sum runs over the samples where both exist. A positive k pairs a with a later
  part of b: a positive lag means b is a later copy of a.
- Output: a tuple `(lag_s, cc_peak, lags_s, cc)`: `lags_s` = k x `dt_s` and `cc` = $c_k$ for k = -K, ..., K (two
  1-D numpy arrays); `cc_peak`, the largest value of `cc` (the largest value, not the largest size); `lag_s`, the
  lag of that value moved to the top of the parabola through it and its two neighbors (in s; at the first or last
  lag there is no parabola: give that lag itself).
- Example: a = [0, 1, 0, 0, 0], b = [0, 0, 1, 0, 0], `dt_s` = 0.01, `max_lag_s` = 0.02 give `cc` = [-0.1, -0.3,
  -0.25, 0.95, -0.35] at the lags -0.02 ... 0.02 s, `cc_peak` = 0.95 and `lag_s` = 0.0098 s (the parabola puts a
  single spike 0.02 samples short of its whole-sample shift).

**Done when** `st.check("M41-T2", func=xcorr_lag)` passes on 24 made-up pairs of windows (three wavelet shapes,
shifts of fractions of a sample, offsets and scaling, noise, inverted copies, shifts beyond the search window):
correlation values against ObsPy's `correlate`, the lag against the top of a parabola fitted by a library, and on
clean cases also against the true shift.

**Stuck?** `st.hint("M41-T2", 1)`

In [ ]:
# Before you code: a and b are windows sampled at 100 Hz, and b is a copy of a delayed by 0.0137 s. At which whole-
# sample lag will the largest value of your correlation sit, and how close to 0.0137 s do you expect the parabola to
# get? If b were an upside-down copy of a, what would the largest value of the correlation look like?
st.predict("M41-T2", """

""")

In [ ]:
def xcorr_lag(a, b, dt_s, max_lag_s):
    """Lag of b against a by normalized cross-correlation, refined below one sample by a parabola.

    a, b      : 1-D numpy arrays of equal length, sampled every dt_s seconds; unchanged
    max_lag_s : largest lag searched (s); K = round(max_lag_s / dt_s) whole samples on each side
    returns   : (lag_s, cc_peak, lags_s, cc): the refined lag (s, positive when b is later), the largest value of cc,
                the lags k * dt_s for k = -K..K and the correlation coefficient at each
    """
    # Step 1: x, y = a and b minus their means; K = round(max_lag_s / dt_s).
    # Step 2: for k = -K..K the sum of x[i] * y[i + k] over the overlap, divided by sqrt(sum x^2 * sum y^2).
    # Step 3: j = position of the largest value. Unless j is the first or last lag, fit the parabola through the
    #         values at j - 1, j, j + 1: its top lies (c[j-1] - c[j+1]) / (2 (c[j-1] - 2 c[j] + c[j+1])) samples
    #         from j.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T2: write your code here")

In [ ]:
st.check("M41-T2", func=xcorr_lag)

Your function on a made-up wavelet sampled at 100 Hz (a Ricker wavelet, the second derivative of a bell curve,
with a peak frequency of 8 Hz; glossary `wavelet`), shifted by 0.0137 s, and on an upside-down copy:

In [ ]:
t_win = np.arange(-0.6, 0.6, 0.01)


def ricker(t, peak_hz):
    """A Ricker wavelet with its peak at t = 0."""
    arg = (np.pi * peak_hz * t) ** 2
    return (1.0 - 2.0 * arg) * np.exp(-arg)


wave_a = ricker(t_win, 8.0)
for label, wave_b in (("delayed copy", ricker(t_win - 0.0137, 8.0)),
                      ("upside-down copy", -ricker(t_win - 0.0137, 8.0))):
    lag_s, cc_peak, lags_s, cc = xcorr_lag(wave_a, wave_b, 0.01, 0.3)
    whole = lags_s[int(np.argmax(cc))]
    print(f"{label}: your lag {lag_s * 1000:.3f} ms (best whole sample {whole * 1000:.0f} ms), peak coefficient "
          f"{cc_peak:.4f}; the most negative coefficient {cc.min():.4f} at {lags_s[int(np.argmin(cc))] * 1000:.0f} ms")

lag_s, cc_peak, lags_s, cc = xcorr_lag(wave_a, ricker(t_win - 0.0137, 8.0), 0.01, 0.3)
fig, ax = plt.subplots(figsize=(6.0, 3.4))
ax.plot(lags_s * 1000.0, cc, "o-", color="tab:blue", ms=3, lw=1.0, label="Your correlation coefficients")
ax.axvline(lag_s * 1000.0, color="k", lw=1.0, label=f"Your lag ({lag_s * 1000:.2f} ms)")
ax.axvline(13.7, color="tab:red", lw=1.0, ls="--", label="True shift (13.7 ms)")
ax.set_xlim(-120, 150)
ax.set_xlabel("Lag of b against a (ms)")
ax.set_ylabel("Correlation coefficient")
ax.set_title("A delayed Ricker wavelet, 100 samples per second", fontsize=10)
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "correlation_function.png", dpi=150)

**Explain (3-5 sentences):** How close did your parabola get to the 0.0137 s shift, compared with the best whole
sample (printed above)? What did the upside-down copy give for the lag and the peak coefficient, and why must a
measurement like that not enter a relocation? In explorer E6 below, make the band narrow and add noise: when does
the lag jump by a whole period, and how does a shorter search window (`max_lag_s`) help?

In [ ]:
# Optional explorer E6 (not checked): a wavelet and its shifted copy, a filter band, noise and a search window;
# your own xcorr_lag computes every correlation and lag that it draws.
st.widgets.lag_explorer(xcorr=xcorr_lag)

## Part 3 · The double-difference system (about 30 min, then task 3)

Place the events in a local frame: x east, y north, z depth (down), all in km. Event i has a position
$\mathbf{x}_i$ and an origin time. If we move it by $\Delta\mathbf{x}_i$ and shift its origin time by
$\Delta\tau_i$, its arrival at station k moves by about

$$\Delta t_i^k \approx \frac{\partial T_i^k}{\partial \mathbf{x}_i}\cdot\Delta\mathbf{x}_i + \Delta\tau_i .$$

The **partial derivatives** (glossary `partial_derivative`) say how much the travel time changes when the source
moves a little east, north or down while everything else stays fixed. Replacing the curved travel-time function by
this straight tangent is a **linearization** (glossary `linearization`). The **double difference** of a pair at
station k is the observed minus the predicted differential time,
$r_k^{ij} = dt_k^{ij,\,\mathrm{obs}} - (T_i^k - T_j^k)$, and it must be explained by the moves of the two events
(Waldhauser & Ellsworth 2000, their equations 3 and 4):

$$\frac{\partial T_i^k}{\partial \mathbf{x}_i}\cdot\Delta\mathbf{x}_i + \Delta\tau_i
  - \frac{\partial T_j^k}{\partial \mathbf{x}_j}\cdot\Delta\mathbf{x}_j - \Delta\tau_j = r_k^{ij}.$$

One such equation per observation gives the system $G\,m = d$: G holds the partial derivatives (it is the
**Jacobian**, glossary `jacobian`, of module M31), m the four changes of every event ($\Delta x$, $\Delta y$,
$\Delta z$, $\Delta\tau$) and d the double differences. In a **half-space** (glossary `half_space`) with one speed
v the rays are **straight rays** (glossary `straight_ray`), $T = |\mathbf{x}_i - \mathbf{s}_k| / v$ for a station at
$\mathbf{s}_k$, and the derivatives are $(\mathbf{x}_i - \mathbf{s}_k) / (v\,|\mathbf{x}_i - \mathbf{s}_k|)$: the
unit vector from the station to the source, divided by the speed.

### M41-T3 · The double-difference system · core · about 120 min

**Goal:** write `dd_system(events_km, stations_km, obs, vp, vs)`: the matrix G and the vector d of the
double-difference equations of a cluster in a homogeneous medium.

**Given:** Part 3 and numpy. Not allowed: numerical or automatic differentiation (`scipy.optimize`,
`scipy.differentiate`, `numdifftools`, `torch`, `jax`, `autograd`) and the course package: write the derivatives
out.

**Contract:**
- Inputs: `events_km`, a numpy array of shape (N, 3) with the current x (east), y (north), z (depth, down) of the
  N events in km; `stations_km`, an array of shape (K, 3) of the stations in the same frame (z down: a station
  above sea level has a negative z); `obs`, a DataFrame with the columns `ev1`, `ev2` (row numbers into
  `events_km`), `sta` (row number into `stations_km`), `phase` (`"P"` or `"S"`) and `dt_s` (observed differential
  time, s), in any order and with any index; `vp`, `vs`, the speeds (km/s). Change none of the inputs.
- Output: a tuple `(G, d)`. G is a numpy array of shape (len(obs), 4N): row r belongs to row r of `obs` (in its
  order); event i owns the columns 4i, 4i+1, 4i+2, 4i+3 for its changes of x, y, z (km) and origin time (s). For
  an observation of the pair (i, j) at station k: event i's columns hold $\partial T_i^k/\partial x$,
  $\partial T_i^k/\partial y$, $\partial T_i^k/\partial z$ (s/km) and +1, event j's columns hold minus the same of
  event j, every other column 0. d is a 1-D array: `dt_s` minus the predicted $T_i^k - T_j^k$ (s), with straight
  rays at the speed of the row's phase.
- Example: one station at (0, 0, 0), events at (0, 0, 10) and (0, 0, 12), vp = 6 km/s, one P row with dt_s = -0.30
  s: G = [[0, 0, 1/6, 1, 0, 0, -1/6, -1]] and d = [-0.30 - (10/6 - 12/6)] = [0.0333] s.

**Done when** `st.check("M41-T3", func=dd_system)` passes on 10 made-up clusters (3-7 events, 4-11 stations at
4-70 km, some above sea level, P and S, pairs in both orders): G against finite differences of the travel time, d
against the made-up data, and G times the known change that made the data must reproduce d.

**Stuck?** `st.hint("M41-T3", 1)`

In [ ]:
# Before you code: a made-up cluster of five events within about 1 km, 10 km deep, and eight stations 15-40 km
# away. Moving one event 100 m east changes some differential times by roughly how much? And moving the whole
# cluster 100 m east? And making every origin time 0.1 s later?
st.predict("M41-T3", """

""")

In [ ]:
def dd_system(events_km, stations_km, obs, vp, vs):
    """Double-difference equations G m = d of a cluster in a homogeneous medium (straight rays).

    events_km   : (N, 3) array of x east, y north, z depth (km) of the events; unchanged
    stations_km : (K, 3) array of the stations in the same frame (z down); unchanged
    obs         : DataFrame with ev1, ev2 (rows of events_km), sta (row of stations_km), phase, dt_s; unchanged
    vp, vs      : speeds of P and S (km/s)
    returns     : (G, d): G of shape (len(obs), 4N) with columns 4i..4i+3 = dx, dy, dz (s/km), dtau (1) of event
                  i; d = dt_s - (T_ev1 - T_ev2) (s)
    """
    # Step 1: for every row: v = vp or vs; r_i = event ev1 minus station sta, r_j = event ev2 minus station sta;
    #         their lengths R_i, R_j; predicted times R / v.
    # Step 2: event ev1's columns: r_i / (v R_i) and +1; event ev2's columns: -(r_j / (v R_j)) and -1.
    # Step 3: d = dt_s - (R_i - R_j) / v.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T3: write your code here")

In [ ]:
st.check("M41-T3", func=dd_system)

Your function on a made-up cluster: five events within about 1 km at 10 km depth, eight stations 15-40 km away
all around, every pair at every station for P and S. The cell compares the changes of d (in ms) that three moves
would cause, through your G, and prints the **singular values** (glossary `singular_value`) of G: how strongly G
stretches each direction of the unknowns. A direction with a singular value of zero changes no differential time
at all; it lies in the **null space** (glossary `null_space`) of G.

In [ ]:
toy_rng = np.random.default_rng(41)
toy_events = np.column_stack([toy_rng.uniform(-0.5, 0.5, 5), toy_rng.uniform(-0.5, 0.5, 5),
                              10.0 + toy_rng.uniform(-0.5, 0.5, 5)])
toy_az = np.radians(np.arange(8) * 45.0 + 10.0)
toy_dist = np.array([15.0, 22.0, 30.0, 40.0, 18.0, 26.0, 35.0, 20.0])
toy_stations = np.column_stack([toy_dist * np.sin(toy_az), toy_dist * np.cos(toy_az), np.zeros(8)])
toy_obs = pd.DataFrame([{"ev1": i, "ev2": j, "sta": k, "phase": ph, "dt_s": 0.0}
                        for i in range(5) for j in range(i + 1, 5) for k in range(8) for ph in ("P", "S")])
G_toy, _ = dd_system(toy_events, toy_stations, toy_obs, 6.0, 3.47)
moves = {"event 0 moved 100 m east": np.zeros(20), "every event moved 100 m east": np.zeros(20),
         "every origin time 0.1 s later": np.zeros(20)}
moves["event 0 moved 100 m east"][0] = 0.1
moves["every event moved 100 m east"][0::4] = 0.1
moves["every origin time 0.1 s later"][3::4] = 0.1
for label, change in moves.items():
    print(f"{label:>32}: largest change of a differential time {1000 * np.abs(G_toy @ change).max():8.3f} ms")
singular = np.linalg.svd(G_toy, compute_uv=False)
print(f"\nG has {G_toy.shape[0]} rows and {G_toy.shape[1]} columns; its singular values, largest first:")
print(np.array2string(singular, precision=4, max_line_width=110))

**Explain (3-5 sentences):** Use the printed numbers. How much did moving one event change the differential
times, and how much moving the whole cluster? Why is a shift of the whole cluster almost invisible to differential
times (think of the rays from two nearby events to a distant station), and which change is exactly invisible?
How many singular values of G are zero or tiny, and which changes do they belong to? What does this mean for the
absolute position of a relocated cluster?

## Part 4 · Solving: damping and a fixed centroid (about 30 min, then task 4)

Least squares (glossary `least_squares`) chooses the m that makes $|G\,m - d|^2$ smallest. Part 3 showed why this
alone is unstable here: the common moves of the cluster are (almost) invisible, so tiny errors in d can move the
whole cluster by kilometres. The ratio of the largest to the smallest singular value, the **condition number**
(glossary `condition_number`), measures how unstable an inversion is. Two remedies, both in HypoDD:

- **Damping** (glossary `damping`): **damped least squares** (glossary `damped_least_squares`) makes
  $|G\,m - d|^2 + \lambda^2 |m|^2$ smallest, so a large change costs something. In equation form: add the rows
  $\lambda I$ below G, with zeros on the right-hand side.
- **A fixed centroid**: the **mean-shift constraint** (glossary `mean_shift_constraint`) asks that the changes of
  all events add up to zero, for x, y, z and the origin time separately, so that the **cluster centroid**
  (glossary `cluster_centroid`), the mean position of the events, stays where it started. In equation form: four
  more rows, row c with a 1 in column 4e + c of every event e, zeros on the right; multiplied by a large weight
  they hold almost exactly.

$$\begin{pmatrix} G \\ \lambda I \\ w\,C \end{pmatrix} m = \begin{pmatrix} d \\ 0 \\ 0 \end{pmatrix}$$

HypoDD solves small clusters by **singular value decomposition** (glossary `svd`) with these four mean-shift rows
(its source file `lsfit_svd.f` adds them with a weight of 1), and large clusters with **LSQR** (glossary `lsqr`;
Paige & Saunders 1982) and damping; PocketQuake tunes LSQR's damping until the condition number lies between 40
and 80, the range the HypoDD manual recommends.

### M41-T4 · Damped least squares with a fixed centroid · core · about 90 min

**Goal:** write `dd_solve(G, d, damp)`: the damped least-squares solution of G m = d whose mean shift of x, y, z and
origin time over all events is zero.

**Given:** Part 4 and numpy (`numpy.linalg.lstsq` solves a stacked system; `scipy.linalg` is allowed). Not allowed:
solvers that damp for you or take constraints for you (`scipy.sparse.linalg`, `scipy.optimize`, `sklearn`,
`statsmodels`, `torch`, `cvxpy`) and the course package.

**Contract:**
- Inputs: `G`, a numpy array of shape (M, 4N) in the column order of task 3; `d`, a 1-D array of length M; `damp`
  (λ), a number of at least 0. Change neither array.
- Output: m, a 1-D numpy array of length 4N, the m that makes $|G\,m - d|^2 + \lambda^2 |m|^2$ smallest among all m
  whose four mean shifts (the means over the events of their x, y, z and origin-time changes) are zero; within
  $10^{-6}$ of its largest value. Four constraint rows weighted $10^4$ times the largest value of G are accurate
  enough; so is an exact solution of the constrained problem.
- Example: two events with every unknown observed directly (G the identity matrix of size 8), d = [1, 0, 0, 0, 0,
  0, 0, 0] and damp = 0 give m = [0.5, 0, 0, 0, -0.5, 0, 0, 0]: the move that fits best while the mean x stays 0.

**Done when** `st.check("M41-T4", func=dd_solve)` passes on 10 made-up systems (3-6 events): exact data of a known
change with zero mean shift (your solution must give it back), noisy data with several dampings, and a change that
also moves the whole cluster; compared with a solution by another method.

**Stuck?** `st.hint("M41-T4", 1)`

In [ ]:
# Before you code: with the made-up cluster of Part 3 and exact data of a change with zero mean shift, tiny damping,
# will your solver give the change back? Now the true change also moves every event 300 m east: what will your
# solution do with those 300 m, and how large will its misfit be?
st.predict("M41-T4", """

""")

In [ ]:
def dd_solve(G, d, damp):
    """Damped least-squares solution of G m = d whose mean shift (x, y, z, origin time) over all events is zero.

    G    : (M, 4N) array (columns of task 3); unchanged
    d    : (M,) array; unchanged
    damp : damping lambda >= 0
    returns : (4N,) array m minimizing |G m - d|^2 + damp^2 |m|^2 with zero mean shift
    """
    # Step 1: n = number of columns; the identity of size n times damp, with n zeros on the right-hand side.
    # Step 2: four rows that add up the x, y, z and origin-time changes over the events, times a large weight
    #         (for example 1e4 times the largest |G|), with four zeros on the right-hand side.
    # Step 3: stack G, the damping rows and the four rows; solve the stacked system by least squares.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T4: write your code here")

In [ ]:
st.check("M41-T4", func=dd_solve)

Your solver on the made-up cluster of Part 3: a change with zero mean shift (exact data), the same change plus a
common move of 300 m east, and the first case with growing damping.

In [ ]:
true_change = np.column_stack([toy_rng.normal(0.0, 0.08, (5, 3)), toy_rng.normal(0.0, 0.01, 5)])
true_change -= true_change.mean(axis=0)                     # zero mean shift
shifted = true_change + np.array([0.3, 0.0, 0.0, 0.0])      # plus 300 m east for every event
for label, change in (("zero mean shift", true_change), ("plus 300 m east", shifted)):
    data = G_toy @ change.ravel()
    found = dd_solve(G_toy, data, 1e-6).reshape(-1, 4)
    shape_err = (found[:, :3] - found[:, :3].mean(0)) - (change[:, :3] - change[:, :3].mean(0))
    misfit_ms = 1000 * np.sqrt(np.mean((G_toy @ found.ravel() - data) ** 2))
    print(f"{label}: mean shift found (x, y, z in m) {np.round(1000 * found[:, :3].mean(0), 2)}; largest shape "
          f"error {1000 * np.abs(shape_err).max():.4f} m; RMS misfit {misfit_ms:.3f} ms")
data = G_toy @ true_change.ravel()
for lam in (0.0, 0.01, 0.1, 1.0):
    found = dd_solve(G_toy, data, lam)
    misfit_ms = 1000 * np.sqrt(np.mean((G_toy @ found - data) ** 2))
    print(f"damp {lam:5.2f}: length of the solution {np.linalg.norm(found):.4f} (true "
          f"{np.linalg.norm(true_change):.4f}), RMS misfit {misfit_ms:.3f} ms")

**Explain (3-5 sentences):** Did your solver give back the zero-mean change? What did it do with the common 300 m,
and how large was the misfit then (printed)? Why can no set of differential times recover such a common move, and
what does your solver put in its place? How did a larger damping change the length of the solution and the misfit,
and when would you want damping in spite of that?

## Part 5 · Rounds of linearized solutions (about 20 min, then task 5)

G holds the slopes of the travel times at the current positions. After a solve the events move, the slopes change,
and the linear prediction of Part 3 is no longer exact. So the solution is found in rounds, an **iteration**
(glossary `iteration`): linearize at the current positions, solve, move the events, and repeat until the
**RMS residual** (glossary `rms_residual`) stops falling. HypoDD runs a fixed number of rounds in each of its
weighting sets.

Two details matter. First, `dt_s` counts from the catalog origin times. Once a round has shifted the origin time of
event i by $\tau_i$, the predicted differential time of a pair is $T_i - T_j + \tau_i - \tau_j$; the simplest way
to keep using your `dd_system` is to subtract $\tau_i - \tau_j$ from `dt_s` before each round. Second, not every
observation is equally good: an **observation weight** (glossary `observation_weight`) multiplies its row of G and
its value of d, so that a weight of 0 removes it. ph2dt gives every catalog differential time a weight from the
quality of its two picks, and HypoDD multiplies it by a weight for P and another for S.

### M41-T5 · Iterate to convergence · core · about 90 min

**Goal:** write `relocate(events_km, stations_km, obs, vp, vs, damp, n_iter, system, solve)`: `n_iter` rounds of
linearized double-difference solutions, with the system and the solver handed in.

**Given:** Part 5, and two functions called like your `dd_system` and `dd_solve` (use the ones you are given, so
that the check can hand in its own). Not allowed: `scipy.optimize`, `torch` and the course package.

**Contract:**
- Inputs: `events_km`, `stations_km`, `obs`, `vp`, `vs` as in task 3 (`obs` may also have a column `weight`, at
  least 0; without it every weight is 1); `damp` as in task 4; `n_iter`, the number of rounds (at least 1);
  `system(positions, stations_km, obs, vp, vs) -> (G, d)` and `solve(G, d, damp) -> m`. Change none of the inputs.
- In every round: subtract the origin shifts found so far ($\tau_{ev1} - \tau_{ev2}$) from `dt_s`, call `system`
  at the current positions, multiply each row of G and its value of d by the row's weight, call `solve`, and add
  the changes to the positions (x, y, z) and to the origin shifts.
- Output: a tuple `(positions_km, origin_shifts_s, rms_s)`: the final positions (shape (N, 3)), the total origin
  shift of every event (shape (N,)), and n_iter + 1 RMS values: `rms_s[k]` is the root of the mean of
  (weight x d)$^2$ over all rows after k rounds (so `rms_s[0]` is at the start positions).
- Example: with exact data (made from known positions and origin shifts), start positions a few hundred metres
  off with a zero mean and enough rounds, the positions land on the known ones and the last RMS is almost 0.

**Done when** `st.check("M41-T5", func=relocate)` passes on 6 made-up clusters (4-7 events, 6-8 rounds), some with
weights where rows of weight 0 hold wrong times; the check hands in its own system and solver, and compares your
result with the positions and origin shifts the data were made from.

**Stuck?** `st.hint("M41-T5", 1)`

In [ ]:
# Before you code: a made-up cluster starts 300 m away from the truth (zero mean). How many rounds do you expect
# until the RMS stops changing, and why more than one? What happens to the origin shifts if you forget to subtract
# the shifts found so far? On the Sangju data of Part 6: will your relocation agree with HypoDD's better
# horizontally or in depth, and what could make them differ?
st.predict("M41-T5", """

""")

In [ ]:
def relocate(events_km, stations_km, obs, vp, vs, damp, n_iter, system, solve):
    """n_iter rounds of linearized double-difference solutions.

    events_km, stations_km, obs, vp, vs : as for dd_system; obs may have a column weight; all unchanged
    damp    : damping for solve
    n_iter  : number of rounds
    system  : function system(positions, stations_km, obs, vp, vs) -> (G, d), like your dd_system
    solve   : function solve(G, d, damp) -> m, like your dd_solve
    returns : (positions_km (N, 3), origin_shifts_s (N,), rms_s (n_iter + 1 values))
    """
    # Step 1: positions = a copy of events_km; shifts = zeros; w = the weight column or ones.
    # Step 2: each round: obs with dt_s - (shift[ev1] - shift[ev2]); G, d from system at the positions; record the
    #         RMS of w * d; m = solve(G * w per row, d * w, damp) as rows of four; add x, y, z and the origin part.
    # Step 3: after the last round, the system once more for the final RMS.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T5: write your code here")

In [ ]:
st.check("M41-T5", func=relocate)

Your three functions together on the made-up cluster of Part 3: the true positions are the toy positions moved by
`true_change` of Part 4, the start positions are the toy positions, and the data are exact.

In [ ]:
truth_xyz = toy_events + true_change[:, :3]
exact = toy_obs.copy()
G_now, d_now = dd_system(truth_xyz, toy_stations, exact.assign(dt_s=0.0), 6.0, 3.47)
exact["dt_s"] = -d_now + (true_change[exact["ev1"], 3] - true_change[exact["ev2"], 3])   # exact differential times
for rounds in (1, 2, 3, 5):
    found, shifts, rms = relocate(toy_events, toy_stations, exact, 6.0, 3.47, 0.0, rounds, dd_system, dd_solve)
    print(f"{rounds} round(s): largest distance from the truth {1000 * np.abs(found - truth_xyz).max():9.4f} m, "
          f"origin shifts off by at most {1000 * np.abs(shifts - true_change[:, 3]).max():.4f} ms; "
          f"RMS by round (ms): {np.round(1000 * rms, 4).tolist()}")

Optional explorer E7 (not checked), next cell: a toy cluster with a few stations. Choose the pick errors, an error
common to all events at each station (a station delay) and the number of stations; your own `dd_system`,
`dd_solve` and `relocate` relocate the events, and the explorer shows how far the absolute positions (from one
linearized absolute location per event) and your relative positions move.

In [ ]:
st.widgets.dd_toy(system=dd_system, solve=dd_solve, relocate=relocate)

## Part 6 · The Sangju earthquakes: HypoDD and your own functions (about 120 min)

PocketQuake's relocation stages run HypoDD on your cluster, starting from the HYPOINVERSE locations of module M32
(kim1983):

1. `ph2dt`: the phases of the HYPOINVERSE archive file go into ph2dt, which pairs the events and writes the catalog
   differential times `dt.ct` (folder `2.HypoDD/00.ph2dt`);
2. `dtct`: HypoDD relocates with `dt.ct` alone (folder `01.dt.ct`);
3. `rereference` and `xcorr`: the waveforms get the HYPOINVERSE origin times, and every event pair is
   cross-correlated at every common station (P on the vertical component, S on the better of the two horizontal
   ones), keeping the times with a coefficient of at least 0.7 in `dt.cc_0.7_combined` (folder `02.dt.cc`);
4. `dtcc`: HypoDD relocates with `dt.ct` and `dt.cc` together, the catalog times leading in the first rounds and
   the cross-correlation times in the later ones.

HypoDD itself computes travel times in the layered model (kim1983) and takes the **first arrival** (glossary
`first_arrival`), the direct wave or a **head wave** (glossary `head_wave`) along a boundary below the source,
whichever comes first. Your `dd_system` knows only straight rays at one speed. For a source in the top layer of
kim1983 (0-15 km; 5.98 km/s for P, 3.40 km/s for S), a straight ray at those speeds is exactly the direct wave. But
the Sangju sources lie close to the boundary at 15 km (their HYPOINVERSE depths are printed below), and beyond the
**crossover distance** (glossary `crossover_distance`; your table of module M30, Part 3) the head wave along that
boundary arrives first: that first arrival leaves the source downward, not upward, and its partial derivatives
differ from those of a straight ray. The crossover shrinks fast as a source nears 15 km. A source below 15 km has
no head wave from that boundary, but its rays start in the faster layer below, so a straight ray in the top layer
only approximates them. Before your own relocation, the notebook therefore works out from your M30 functions which
stations each phase may use, and measures that approximation. Moho head waves (Pn and Sn) are not first arrivals
within 100 km for these depths (M30), and PocketQuake's station list ends at 100 km, so no location here uses them.

Your cluster first: it must hold the HYPOINVERSE locations of module M32, made from picks with PocketQuake's
default threshold.

In [ ]:
arc_files = sorted((RUN_DIR / "1.HypoInv" / "kim1983").glob("*.arc"))
if not arc_files:
    raise FileNotFoundError(f"PocketQuake has no HYPOINVERSE locations of cluster {PQ_SLUG!r} under {RUN_DIR}. Part 6 "
                            "starts from them: finish task M32-T1 of module M32 first (or set PQ_SLUG to the cluster "
                            "you located there).")
REGION = arc_files[0].stem                       # PocketQuake's name of your cluster in its file names
hypo = st.pq.read_sum(PQ_SLUG, "kim1983")
print(f"HYPOINVERSE (kim1983) of cluster {PQ_SLUG!r}, from {hypo.attrs['source']}:")
print(hypo[["id", "time", "lat", "lon", "depth", "num", "rms", "erh", "erz", "qual"]].to_string(index=False))
lowest = min(pd.read_csv(f)["Probability"].min() for f in sorted((RUN_DIR / "picks").glob("*_picks.csv")))
print(f"\nlowest pick probability in your cluster's pick files: {lowest:.3f} (PocketQuake's default threshold: 0.3)")
if lowest >= 0.5:
    print("Note: these picks come from a raised threshold. Rerun 'picking' to 'hypoinverse' with the default "
          "threshold first, as in module M32.")

The relocation stages: a **dry run** (glossary `dry_run`) first, which only prints what would happen; read it, then
run the next cell. `ph2dt` and `dtct` take a second or two.

In [ ]:
st.pq.run(PQ_SLUG, "ph2dt", "dtct", cores=PQ_CORES, mem_gb=PQ_MEM_GB)          # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "ph2dt", "dtct", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

The cross-correlation stage is the long one: 15 event pairs at about 90 stations, every measurement on traces
interpolated to 1000 samples per second. It takes about 8 minutes on 3 cores (start it, then read on).

In [ ]:
assert st.pq.run(PQ_SLUG, "rereference", "xcorr", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

In [ ]:
assert st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
COPIES = Path(RESULT_DIR) / "hypodd"


def copy_dtcc_products(folder):
    """Copy the products in the dt.cc folder that the last logged run wrote into results/hypodd/<folder>/."""
    last = json.loads(RUNS_LOG.read_text(encoding="utf-8").splitlines()[-1])
    copies = []
    for rel in last["products"]:
        if rel.startswith("2.HypoDD/02.dt.cc/"):
            copy = COPIES / folder / Path(rel).relative_to("2.HypoDD/02.dt.cc")
            copy.parent.mkdir(parents=True, exist_ok=True)
            copies.append(Path(shutil.copy2(RUN_DIR / rel, copy)))
    return copies


first_copies = copy_dtcc_products("first")
print("copied the dt.cc products of this run:", [str(p) for p in first_copies])
for line in RUNS_LOG.read_text(encoding="utf-8").splitlines()[-3:]:
    run = json.loads(line)
    print(run["ts"], f"{run['stage_from']} -> {run['through']}", f"{run['seconds']:.0f} s", run["products"])

Readers for HypoDD's text files (not tasks; each prints nothing and changes nothing):

In [ ]:
KM_PER_DEG = 6371.0 * np.pi / 180.0


def read_pha(path):
    """ph2dt's phase file: (picks, origins). '#' lines start an event (date, time, ..., id); 'STA TT WGHT PHA' lines
    are its picks, TT being the travel time from the HYPOINVERSE origin. Times in s after the first origin."""
    rows, starts, event = [], {}, None
    for line in Path(path).read_text().splitlines():
        part = line.split()
        if part and part[0] == "#":
            event = part[-1]
            starts[event] = UTCDateTime(*(int(v) for v in part[1:6])) + float(part[6])
        elif part:
            rows.append({"event_id": event, "station": part[0], "tt": float(part[1]), "weight": float(part[2]),
                         "phase": part[3]})
    zero = min(starts.values())
    origins = {e: t - zero for e, t in starts.items()}
    table = pd.DataFrame(rows)
    table["time_s"] = table["tt"] + table["event_id"].map(origins)
    return table, origins


def read_dt(path):
    """A dt.ct ('STA T1 T2 WGHT PHA') or dt.cc ('STA DT CC PHA') file: one row per observation, with its pair."""
    rows, pair = [], None
    for line in Path(path).read_text().splitlines():
        part = line.split()
        if part and part[0] == "#":
            pair = part[1:3]
        elif len(part) == 5:
            rows.append({"ev1": pair[0], "ev2": pair[1], "station": part[0], "dt_s": float(part[1]) - float(part[2]),
                         "weight": float(part[3]), "phase": part[4]})
        elif len(part) == 4:
            rows.append({"ev1": pair[0], "ev2": pair[1], "station": part[0], "dt_s": float(part[1]),
                         "cc": float(part[2]), "phase": part[3]})
    return pd.DataFrame(rows)


def read_table(path, names):
    """A whitespace-separated table without a header (event.dat, station.dat)."""
    return pd.read_csv(path, sep=r"\s+", header=None, names=names, usecols=range(len(names)), dtype={"id": str})


def flat_km(lat, lon, lat0, lon0):
    """East and north (km) of points from (lat0, lon0) on a flat local map (6371-km sphere)."""
    east = (np.asarray(lon, dtype=float) - lon0) * KM_PER_DEG * np.cos(np.radians(lat0))
    return east, (np.asarray(lat, dtype=float) - lat0) * KM_PER_DEG


events_dat = read_table(DD_DIR / "00.ph2dt" / "event.dat",
                        ["date", "hhmmss", "lat", "lon", "depth_km", "mag", "eh", "ez", "rms", "id"])
stations_dat = read_table(DD_DIR / "00.ph2dt" / "station.dat", ["station", "lat", "lon"])
print(f"event.dat: {len(events_dat)} events; station.dat: {len(stations_dat)} stations; ph2dt.inp settings:")
print("\n".join((DD_DIR / "00.ph2dt" / "ph2dt.inp").read_text().splitlines()[-2:]))

### Your catalog differential times against ph2dt

ph2dt read the phase file `<Region>_mod.pha` next to `dt.ct`. Its settings (above) include MINWGHT = 0: picks with
a negative weight are not used. The cell runs your `catalog_dt` on the same phases and compares every row of
`dt.ct` with yours.

In [ ]:
pha_picks, pha_origins = read_pha(DD_DIR / "00.ph2dt" / f"{REGION}_mod.pha")
used = pha_picks[pha_picks["weight"] >= 0.0]
mine_ct = catalog_dt(used, pha_origins)
ct = read_dt(DD_DIR / "00.ph2dt" / "dt.ct")
both_ct = ct.merge(mine_ct, on=["ev1", "ev2", "station", "phase"], how="left", suffixes=("_ph2dt", "_yours"))
print(f"phase file: {len(pha_picks)} picks of {len(pha_origins)} events, {len(used)} with a weight of 0 or more")
print(f"dt.ct: {len(ct)} differential times of {ct[['ev1', 'ev2']].drop_duplicates().shape[0]} pairs; yours: "
      f"{len(mine_ct)} of {mine_ct[['ev1', 'ev2']].drop_duplicates().shape[0]} pairs")
print(f"rows of dt.ct missing from yours: {both_ct['dt_s_yours'].isna().sum()}; largest |ph2dt - yours|: "
      f"{np.nanmax(np.abs(both_ct['dt_s_ph2dt'] - both_ct['dt_s_yours'])):.2e} s")

extra = mine_ct.merge(ct, on=["ev1", "ev2", "station", "phase"], how="left", indicator=True)
extra = extra[extra["_merge"] == "left_only"].copy()
ex, ny = flat_km(events_dat["lat"], events_dat["lon"], events_dat["lat"].mean(), events_dat["lon"].mean())
place = pd.DataFrame({"x": ex, "y": ny, "z": events_dat["depth_km"].to_numpy()}, index=events_dat["id"])
gap = place.loc[extra["ev1"]].to_numpy() - place.loc[extra["ev2"]].to_numpy()
extra["separation_km"] = np.linalg.norm(gap, axis=1)
# ph2dt.f: "remove outliers above the separation-delaytime line": |dt| > separation / v + 0.5 s, v = 4 (P), 2.3 (S)
extra["line_s"] = extra["separation_km"] / np.where(extra["phase"] == "P", 4.0, 2.3) + 0.5
print(f"\nyour rows that ph2dt left out: {len(extra)}; of them above ph2dt's separation-delay line: "
      f"{int((extra['dt_s_x'].abs() > extra['line_s']).sum())}")
print(extra[["ev1", "ev2", "station", "phase", "dt_s_x", "separation_km", "line_s"]]
      .rename(columns={"dt_s_x": "your dt_s"}).round(3).head(8).to_string(index=False))

Such a row is an **outlier** (glossary `outlier`): no two earthquakes a few kilometres apart can differ in travel
time by seconds. Module M32 found the cause at these far stations (an S wave picked as P).

### Your cross-correlation times against PocketQuake's

The re-referenced waveforms in your run folder carry the picks in their **SAC headers** (glossary `sac_header`;
`a` for P, `t0` for S) and the HYPOINVERSE origin as their reference time. The cell cuts a window around each
pick on both records, filters both the same way (`CC_BAND_HZ`), runs your `xcorr_lag`, and turns the lag into a
differential time. If window a starts at $s_a$ and window b at $s_b$ (absolute times) and b lags a by $\ell$,
then $dt = (s_a - t_{0,a}) - (s_b - t_{0,b}) - \ell$: when the windows start exactly at the picks, this is the
catalog differential time minus the lag. PocketQuake measured the same pairs by its own method (traces interpolated to
1000 samples per second, the second window slid over ±0.5 s); the cell compares every time of its `dt.cc` file
with yours.

In [ ]:
cc_pq = read_dt(DD_DIR / "02.dt.cc" / "dt.cc_0.7_combined")
names = sorted(p.name for p in (RUN_DIR / "waveforms_100km").iterdir() if p.is_dir())
folder_of = {}
for row in events_dat.itertuples():                       # the folder whose name is nearest to the origin time
    when = UTCDateTime.strptime(f"{row.date}{int(row.hhmmss):08d}", "%Y%m%d%H%M%S%f")
    folder_of[row.id] = min(names, key=lambda f: abs(UTCDateTime.strptime(f, "%Y%m%d%H%M%S") - when))
cache = {}


def filtered(event, station, comp):
    """The re-referenced record of one event, station (NETSTA) and component, demeaned, tapered and band-passed."""
    key = (event, station, comp)
    if key not in cache:
        folder = folder_of[event]
        files = sorted((RUN_DIR / "waveforms_100km" / folder).glob(f"{folder}.{station[:2]}.{station[2:]}.*{comp}.sac"))
        trace = read(str(files[0]))[0] if files else None
        if trace is not None:
            trace.detrend("demean").taper(0.05)
            trace.filter("bandpass", freqmin=CC_BAND_HZ[0], freqmax=CC_BAND_HZ[1], corners=4, zerophase=True)
        cache[key] = trace
    return cache[key]


def cut(trace, center, before, after):
    """Samples from `before` s ahead of `center` to `after` s past it (a fixed count), and the first sample's time."""
    first = int(round((center - before - trace.stats.starttime) * trace.stats.sampling_rate))
    count = int(round((before + after) * trace.stats.sampling_rate)) + 1
    return trace.data[first:first + count].astype(float), trace.stats.starttime + first * trace.stats.delta


rows = []
for obs_row in cc_pq.itertuples():
    best = None
    for comp in (("Z",) if obs_row.phase == "P" else ("N", "E")):
        tr1, tr2 = filtered(obs_row.ev1, obs_row.station, comp), filtered(obs_row.ev2, obs_row.station, comp)
        if tr1 is None or tr2 is None:
            continue
        mark = "a" if obs_row.phase == "P" else "t0"
        t01 = tr1.stats.starttime - tr1.stats.sac.b         # the reference (origin) time of each record
        t02 = tr2.stats.starttime - tr2.stats.sac.b
        win1, start1 = cut(tr1, t01 + tr1.stats.sac[mark], *CC_WINDOW_S)
        win2, start2 = cut(tr2, t02 + tr2.stats.sac[mark], *CC_WINDOW_S)
        lag_s, cc_peak, _, _ = xcorr_lag(win1, win2, tr1.stats.delta, MAX_LAG_S)
        if best is None or cc_peak > best[1]:
            best = ((start1 - t01) - (start2 - t02) - lag_s, cc_peak)
    if best is not None:
        rows.append({"ev1": obs_row.ev1, "ev2": obs_row.ev2, "station": obs_row.station, "phase": obs_row.phase,
                     "pq_dt_s": obs_row.dt_s, "pq_cc": obs_row.cc, "your_dt_s": best[0], "your_cc": best[1]})
mine_cc = pd.DataFrame(rows)
mine_cc["diff_ms"] = 1000.0 * (mine_cc["your_dt_s"] - mine_cc["pq_dt_s"])
mine_cc["your cc"] = pd.cut(mine_cc["your_cc"], [-1.0, 0.7, 0.8, 0.9, 1.0], labels=["< 0.7", "0.7-0.8", "0.8-0.9",
                                                                                     ">= 0.9"])
summary_cc = mine_cc.groupby(["phase", "your cc"], observed=True)["diff_ms"].agg(
    times="size", median_ms="median", mad_ms=lambda v: (v - v.median()).abs().median(),
    over_10_ms=lambda v: int((v.abs() > 10.0).sum()))
print(f"dt.cc: {len(cc_pq)} times (coefficient >= 0.7 in PocketQuake's measurement); yours for {len(mine_cc)} of them")
print(summary_cc.round(2).to_string())
big = mine_cc[mine_cc["diff_ms"].abs() > 10.0]
print(f"\ndifferences larger than 10 ms: {len(big)}; their median size {big['diff_ms'].abs().median():.0f} ms")
mine_cc.to_csv(Path(RESULT_DIR) / "M41_cc_times.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
    part = mine_cc[mine_cc["phase"] == phase]
    ax.plot(part["your_cc"], part["diff_ms"], "o", ms=3, color=color, alpha=0.7, label=f"{phase}")
ax.set_yscale("symlog", linthresh=1.0)
ax.axhline(0.0, color="0.5", lw=0.8)
ax.set_xlabel("Your peak correlation coefficient")
ax.set_ylabel("Your dt minus PocketQuake's (ms)")
ax.set_title("Cross-correlation times of the Sangju pairs", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "cc_times_vs_pocketquake.png", dpi=150)

### Your relocation against HypoDD's

Which stations can your straight rays use? Paste your `direct_time` and `head_wave` of module M30 into the next
cell as they are in your M30 notebook (they need numpy only), as you did in module M31, and run it.

In [ ]:
# Paste your direct_time and head_wave of module M30 below this line.

The next cell takes the HYPOINVERSE depths of the events that HypoDD's dt.ct run kept (the start of every
relocation below) and finds the deepest one above the boundary at 15 km. With your `head_wave` it computes that
event's crossover distances for P and for S along this boundary and rounds them down to whole kilometres: these are
the station limits of your relocation, one per phase. A shallower source has a larger crossover, so inside these
limits the direct wave is the first arrival of every event above the boundary. A station counts as inside when it
is inside for every one of these events. For an event below the boundary, the cell compares your `direct_time`
(layered) with the straight-ray time at the stations it keeps. It first prints which events HypoDD relocated, and
HypoDD's data-weighting lines: they show how its distance cutoff for catalog times (WDCT, km) shrinks from set to
set.

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if not callable(globals().get(name))]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above "
                    "and run it (finish task M30-T2 of module M30 first; task M30-T1 writes direct_time).")
MODEL = st.data.velocity_model(MODEL_NAME)
BOUNDARY_KM = float(MODEL["top_km"].iloc[1])                     # the boundary below the top layer
dtct_reloc = st.pq.read_reloc(PQ_SLUG, "dtct")
dtcc_reloc = st.pq.read_reloc(PQ_SLUG, "dtcc")
kept = [str(i) for i in dtct_reloc["id"] if i in set(dtcc_reloc["id"])]
print(f"HypoDD relocated {len(dtct_reloc)} (dt.ct) and {len(dtcc_reloc)} (dt.cc) of {len(events_dat)} events; "
      f"compared below: {kept}; left out by dt.ct: {sorted(set(events_dat['id']) - set(map(str, dtct_reloc['id'])))}")
inp_lines = (DD_DIR / "01.dt.ct" / "hypoDD.inp").read_text().splitlines()
at = next(n for n, line in enumerate(inp_lines) if line.startswith("*--- data weighting"))
print("hypoDD.inp (dt.ct):", inp_lines[at].strip())
print("\n".join(inp_lines[at + 1:at + 6]))

lat0, lon0 = events_dat["lat"].mean(), events_dat["lon"].mean()
start_rows = events_dat.set_index("id").loc[kept]
e0, n0 = flat_km(start_rows["lat"], start_rows["lon"], lat0, lon0)
start_km = np.column_stack([e0, n0, start_rows["depth_km"].to_numpy()])
se, sn = flat_km(stations_dat["lat"], stations_dat["lon"], lat0, lon0)
sites_km = np.column_stack([se, sn, np.zeros(len(se))])
reach = np.hypot(sites_km[:, None, 0] - start_km[None, :, 0], sites_km[:, None, 1] - start_km[None, :, 1])
far = reach.max(axis=1)                          # distance of each station from the farthest of the events
row_of = {e: i for i, e in enumerate(kept)}
site_of = {s: i for i, s in enumerate(stations_dat["station"])}
ct_kept = ct[ct["ev1"].isin(row_of) & ct["ev2"].isin(row_of)].reset_index(drop=True)
dd_obs = pd.DataFrame({"ev1": ct_kept["ev1"].map(row_of), "ev2": ct_kept["ev2"].map(row_of),
                       "sta": ct_kept["station"].map(site_of), "phase": ct_kept["phase"], "dt_s": ct_kept["dt_s"],
                       "weight": ct_kept["weight"] * np.where(ct_kept["phase"] == "P", 1.0, 0.5)})

depth_of = start_rows["depth_km"]
upper = depth_of[depth_of < BOUNDARY_KM]
print(f"\nHYPOINVERSE depths (km): {depth_of.round(2).to_dict()}; boundary of {MODEL_NAME} at {BOUNDARY_KM:g} km")
print(f"deepest event above the boundary: {upper.idxmax()} at {upper.max():.2f} km")
LIMIT_KM = {}
for phase in ("P", "S"):
    crossover = float(np.asarray(head_wave(np.array([50.0]), upper.max(), MODEL, phase, 1)[2]))
    LIMIT_KM[phase] = float(np.floor(crossover))
    used = dd_obs[(dd_obs["phase"] == phase) & (far[dd_obs["sta"].to_numpy()] <= LIMIT_KM[phase])]
    print(f"  {phase}: your crossover {crossover:.2f} km -> limit {LIMIT_KM[phase]:g} km; {used['sta'].nunique()} "
          f"stations with {len(used)} differential times inside it")
for event in depth_of[depth_of >= BOUNDARY_KM].index:
    for phase, speed in (("P", VP_KM_S), ("S", VS_KM_S)):
        pair_has = (dd_obs["ev1"] == row_of[event]) | (dd_obs["ev2"] == row_of[event])
        sta = np.unique(dd_obs.loc[pair_has & (dd_obs["phase"] == phase), "sta"].to_numpy())
        sta = sta[far[sta] <= LIMIT_KM[phase]]
        x = np.sort(reach[sta, row_of[event]])
        miss = 1000.0 * (direct_time(x, depth_of[event], MODEL, phase) - np.hypot(x, depth_of[event]) / speed)
        print(f"  event {event} at {depth_of[event]:.2f} km, {phase}: your layered direct time minus the straight "
              f"ray at its {len(x)} stations: {miss[0]:.1f} ms at the nearest ({x[0]:.0f} km), {miss[-1]:.1f} ms at "
              f"the farthest ({x[-1]:.0f} km), from {miss.min():.1f} to {miss.max():.1f} ms in all")

Your relocation: the events that HypoDD's dt.ct run kept, with your three functions, from the same start
(event.dat, the HYPOINVERSE locations) and with the same catalog differential times (`dt.ct`), weighted like
HypoDD weights them (ph2dt's weight times 1.0 for P and 0.5 for S, from `hypoDD.inp`), with straight rays at the
top layer's speeds: once with each phase inside its limit, once with every station. The cell compares your
positions with HypoDD's after removing each set's centroid (the shape), and the centroids themselves; then it
relocates once more without the events below the boundary, to measure how much they change the others.

In [ ]:
results = {}
for label, limits in (("inside the crossover", LIMIT_KM), ("all, to 100 km", {"P": np.inf, "S": np.inf})):
    inside = far[dd_obs["sta"].to_numpy()] <= dd_obs["phase"].map(limits).to_numpy()
    near = dd_obs[inside].reset_index(drop=True)
    found, shifts, rms = relocate(start_km, sites_km, near, VP_KM_S, VS_KM_S, DAMP, N_ITER, dd_system, dd_solve)
    results[label] = found
    print(f"\nyour relocation, stations {label}: {near['sta'].nunique()} stations, {len(near)} differential times; "
          f"weighted RMS by round (ms): {np.round(1000 * rms, 1).tolist()}")


def as_km(table):
    """x east, y north, z depth (km) of a relocation table, in the order of `kept`."""
    rows_ = table.set_index(table["id"].astype(str)).loc[kept]
    e, n = flat_km(rows_["lat"], rows_["lon"], lat0, lon0)
    return np.column_stack([e, n, rows_["depth"].to_numpy()])


def centred(xyz):
    return xyz - xyz.mean(axis=0)


sets = {"HYPOINVERSE": start_km, "HypoDD dt.ct": as_km(dtct_reloc), "HypoDD dt.cc": as_km(dtcc_reloc),
        **{f"yours, {k}": v for k, v in results.items()}}
print("\nshape differences from HypoDD dt.ct (RMS over the events, after removing each centroid; m):")
for label, xyz in sets.items():
    rms_m = 1000.0 * np.sqrt(np.mean((centred(xyz) - centred(sets["HypoDD dt.ct"])) ** 2, axis=0))
    shift_m = 1000.0 * (xyz.mean(axis=0) - start_km.mean(axis=0))
    print(f"  {label:>30}: east {rms_m[0]:6.1f}, north {rms_m[1]:6.1f}, depth {rms_m[2]:6.1f};   centroid minus "
          f"HYPOINVERSE's: {np.round(shift_m, 0)}")
per_event = pd.DataFrame(1000.0 * (centred(results["inside the crossover"]) - centred(sets["HypoDD dt.ct"])),
                         columns=["east_m", "north_m", "depth_m"], index=kept)
print("\nyours (inside the crossover) minus HypoDD dt.ct, per event, after removing each centroid (m):")
print(per_event.round(0).to_string())

below = [e for e in kept if depth_of[e] >= BOUNDARY_KM]
if below:                                        # the same relocation without the events below the boundary
    rest = [i for i, e in enumerate(kept) if e not in below]
    renumber = {old: new for new, old in enumerate(rest)}
    inside = far[dd_obs["sta"].to_numpy()] <= dd_obs["phase"].map(LIMIT_KM).to_numpy()
    near = dd_obs[inside & dd_obs["ev1"].isin(rest) & dd_obs["ev2"].isin(rest)]
    near = near.assign(ev1=near["ev1"].map(renumber), ev2=near["ev2"].map(renumber)).reset_index(drop=True)
    found, shifts, rms = relocate(start_km[rest], sites_km, near, VP_KM_S, VS_KM_S, DAMP, N_ITER, dd_system,
                                  dd_solve)
    print(f"\nyour relocation without {below}: {near['sta'].nunique()} stations, {len(near)} differential times; "
          f"weighted RMS by round (ms): {np.round(1000 * rms, 1).tolist()}")
    with_them = results["inside the crossover"][rest]
    for label, xyz in ((f"with {below}", with_them), (f"without {below}", found)):
        rms_m = 1000.0 * np.sqrt(np.mean((centred(xyz) - centred(sets["HypoDD dt.ct"][rest])) ** 2, axis=0))
        print(f"  the other {len(rest)} events, yours {label}: shape against HypoDD dt.ct east {rms_m[0]:.1f}, "
              f"north {rms_m[1]:.1f}, depth {rms_m[2]:.1f} m")
    change = 1000.0 * (centred(found) - centred(with_them))
    print(f"  leaving {below} out moved the others by at most {np.abs(change[:, :2]).max():.0f} m horizontally and "
          f"{np.abs(change[:, 2]).max():.0f} m in depth (after removing each centroid)")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
styles = {"HYPOINVERSE": ("o", "0.6"), "HypoDD dt.ct": ("s", "tab:blue"), "HypoDD dt.cc": ("D", "tab:green"),
          "yours, inside the crossover": ("^", "tab:red")}
for label, (marker, color) in styles.items():
    xyz = sets[label] - start_km.mean(axis=0)
    axes[0].plot(1000 * xyz[:, 0], 1000 * xyz[:, 1], marker, color=color, ms=6, mfc="none",
                 label=label[0].upper() + label[1:])
    axes[1].plot(1000 * xyz[:, 0], 1000 * xyz[:, 2], marker, color=color, ms=6, mfc="none")
axes[0].set_xlabel("East of the HYPOINVERSE centroid (m)")
axes[0].set_ylabel("North of the HYPOINVERSE centroid (m)")
axes[0].set_aspect("equal", adjustable="datalim")
axes[0].set_title("Map view", fontsize=10)
axes[1].set_xlabel("East of the HYPOINVERSE centroid (m)")
axes[1].set_ylabel("Below the HYPOINVERSE centroid (m)")
axes[1].invert_yaxis()
axes[1].set_title("Depth section, looking north", fontsize=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "relocations.png", dpi=150)

**Explain (3-5 sentences):** Use the numbers printed in Part 6. How closely did your catalog differential times
match ph2dt's, and which rule explains the rows ph2dt left out? For which coefficients did your cross-correlation
times agree with PocketQuake's, and what were the large differences (compare their size with the period of a
5-20 Hz wave)? Which station limits did your crossover distances give, and how many stations did each phase keep?
How far is your relocation from HypoDD's dt.ct relocation in shape, horizontally and in depth, and what changed when
you also used the stations beyond your limits? What do your straight rays miss for the event below 15 km, and how
much did leaving it out change the others? Where did HypoDD put the centroid, compared with the HYPOINVERSE
locations, and why does your relocation leave it where it started?

## Part 7 · Reruns, the bootstrap and a colleague's file (about 90 min, task 6)

**The bootstrap** (glossary `bootstrap`; Efron & Tibshirani 1986) asks how much a result would change if the data
had come out a little differently. PocketQuake's `bootstrap_relocation` draws the differential times again, with
replacement, as many as there are (some twice, some not at all), reruns HypoDD on each such **bootstrap replica**
(glossary `bootstrap_replica`) starting from the relocation, shifts each replica so that its median offset from the
relocation is zero, and reports for every event half the distance between the 2.5th and the 97.5th **percentile**
(glossary `percentile`) of its positions: the 95 % half-width, along x (east), y (north) and z (depth), in metres.
Read how it seeds its **random numbers** (glossary `random_seed`): the docstring lines of your copy are printed
below.

**The run log** (glossary `run_log`): every run through `st.pq` appends one line in JSON Lines (glossary
`json_lines`) to `work/.sgtl/runs.jsonl`: what ran, when it started (`ts`, UTC), its **return code** (glossary
`return_code`, 0 = it worked) and the SHA-256 fingerprint (glossary `checksum`) of every product the run created or
changed. Every file also has a **modification time** (glossary `modification_time`): when its contents were last
written. In module M32 your `run_products` listed what
the runs of one stage logged; task 6 asks the reverse question: given a file as it is now, which run wrote it?

In [ ]:
source = (st.pq.pipeline_dir() / "pipeline" / "core" / "hypodd.py").read_text(encoding="utf-8").splitlines()
for number, line in enumerate(source, 1):
    if "Deterministic: replica" in line or "default_rng(seed + i)" in line or "Resampling is **global**" in line:
        print(f"hypodd.py:{number}: {line.strip()}")

### M41-T6 · Reruns, two bootstraps and a colleague's file · core · about 90 min

**Goal:** write `who_wrote(path, log_path, slug)`: the run of the log that wrote a file as it is now. Then rerun
HypoDD's dt.cc stage unchanged, bootstrap it twice with seeds `BOOT_SEED` and `BOOT_SEED_CHANGED` (the one setting
you change; `BOOT_N` replicas each), and report the runs in a dictionary `dd_runs`.

**Given:** the run cells below (read each dry run before you apply it), the run log, and the colleague's cell.

**Contract (the function):**
- Inputs: `path`, a product file (text or `pathlib.Path`); `log_path`, a run log in JSON Lines (one record per
  line, records like those printed in Part 6); `slug`, a cluster name.
- Output: the record (a dict, as `json.loads` reads it) of the most recent run of cluster `slug` whose `products`
  hold the SHA-256 fingerprint of the file's bytes as they are now (among its values); `None` when no run of that
  cluster logged this fingerprint. Skip lines that are not valid JSON and records without `products`.
- Example: a log with a run of cluster `"toy_abcd"` at `2026-01-01T00:00:00Z` that logged the file's fingerprint and
  a later run of the same cluster that failed with an empty `products` gives the first record.

**Contract (the report):** `dd_runs = {"who_wrote": who_wrote, "slug": PQ_SLUG, "first": {<path of each copy of
the dt.cc products of Part 6's run (first_copies)>: <its SHA-256 fingerprint>}, "rerun": {<path of each copy after
the unchanged rerun (rerun_copies)>: <its SHA-256 fingerprint>}, "boot": <the copy of bootstrap_errors.csv for
BOOT_SEED>, "boot_changed": <the copy for BOOT_SEED_CHANGED>}`. Paths as text; fingerprints as `hexdigest()`
writes them (64 characters).

**Done when** `st.check("M41-T6", answer=dd_runs)` passes: your `who_wrote` on 8 made-up logs (a newer writer, an
identical rerun, a later failed run, a later run of another stage, a file restored from an old copy, a file edited
after the last run, a run of another cluster, a half-written line), then your copies: the check recomputes the
fingerprint of every copy and compares it with yours, the fingerprints must be among those logged for your runs
(a rerun that wrote the same bytes logged as a run of its own), and the two bootstrap tables must differ in the
seed only, with seeds far enough apart to share no replica. It never prints a location or an error value.

**Stuck?** `st.hint("M41-T6", 1)`

In [ ]:
# A colleague's cell. They want to know whether HypoDD's catalog relocation depends on the number of cores, so they
# rerun the dt.ct stage on 8 cores and compare the relocation before and after. The run's own messages are hidden to
# keep their notebook short. Run it as it is.
colleague_before = st.pq.read_reloc(PQ_SLUG, "dtct")
colleague_started = UTCDateTime()
with contextlib.redirect_stdout(io.StringIO()):
    st.pq.run(PQ_SLUG, "dtct", "dtct", cores=8, mem_gb=PQ_MEM_GB, apply=True)
colleague_after = st.pq.read_reloc(PQ_SLUG, "dtct")
moved = (colleague_after[["x", "y", "z"]].to_numpy() - colleague_before[["x", "y", "z"]].to_numpy())
print(f"colleague: run started {colleague_started} (UTC); largest change of a relocated position: "
      f"{np.abs(moved).max():.1f} m")
print("colleague's verdict: HypoDD's catalog relocation does not depend on the number of cores.")

In [ ]:
# Before you code: do you accept the colleague's verdict? Name two things you could look at to find out which run
# wrote the dt.ct hypoDD.reloc the colleague read. Will the unchanged rerun of the dt.cc stage write the same file,
# byte for byte? By how much do you expect the 95 % half-widths to change when only the bootstrap seed changes?
st.predict("M41-T6", """

""")

In [ ]:
def who_wrote(path, log_path, slug):
    """The most recent logged run of cluster `slug` whose products hold the SHA-256 of the file as it is now.

    path     : the product file (str or pathlib.Path)
    log_path : the run log (JSON Lines)
    slug     : the cluster
    returns  : that run's record (dict), or None
    """
    # Step 1: the SHA-256 fingerprint of the file's bytes (hashlib.sha256(...).hexdigest()).
    # Step 2: read the log line by line; skip lines that are not valid JSON (json.loads raises ValueError).
    # Step 3: keep the last record of this cluster whose products (a dict) have the fingerprint among their values.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T6: write your code here")

Run 1: the dt.cc stage again, nothing changed (a dry run first).

In [ ]:
st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB)            # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_copies = copy_dtcc_products("rerun")
print("copied the dt.cc products of the rerun:", [str(p) for p in rerun_copies])

Runs 2 and 3: the bootstrap of the dt.cc relocation, `BOOT_N` replicas, first with `BOOT_SEED`, then with
`BOOT_SEED_CHANGED`. Each run writes `bootstrap_errors.csv` in the dt.cc folder (the second overwrites the first),
so the cells copy it.

In [ ]:
st.pq.bootstrap(PQ_SLUG, "dtcc", n=BOOT_N, seed=BOOT_SEED, cores=PQ_CORES, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
boot_copies = {}
for seed in (BOOT_SEED, BOOT_SEED_CHANGED):
    assert st.pq.bootstrap(PQ_SLUG, "dtcc", n=BOOT_N, seed=seed, cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
    boot_copies[seed] = Path(shutil.copy2(DD_DIR / "02.dt.cc" / "bootstrap_errors.csv",
                                          COPIES / f"bootstrap_seed{seed}.csv"))
    print("copied", boot_copies[seed])

In [ ]:
# Fill in dd_runs. "first" and "rerun" map the path of every copy in first_copies and rerun_copies (as text) to the
# SHA-256 fingerprint of that copy's bytes; the check recomputes each one from the file.
# YOUR CODE HERE
raise NotImplementedError("M41-T6: write your code here")
print("copies in your report:", {k: (sorted(v) if isinstance(v, dict) else v) for k, v in dd_runs.items()
                                 if k in ("first", "rerun", "boot", "boot_changed")})

In [ ]:
st.check("M41-T6", answer=dd_runs)

What the colleague read, with your `who_wrote`: the run that wrote the dt.ct relocation as it is now, the file's
modification time, the last lines of the run log and the newest files of PocketQuake's log folder. A run that sgtl
refuses to start leaves no line in the log and no file in the log folder. A result file left over from a run older
than the one you think you are reading is a **stale file** (glossary `stale_file`).

In [ ]:
dtct_file = DD_DIR / "01.dt.ct" / "hypoDD.reloc"
writer = who_wrote(dtct_file, RUNS_LOG, PQ_SLUG)
print("the colleague's run started:   ", colleague_started)
if writer is None:
    print("no logged run of this cluster wrote the dt.ct relocation as it is now")
else:
    print(f"the run that wrote the file:   started {writer['ts']}, stages {writer.get('stage_from')} -> "
          f"{writer.get('through')}, cores {writer.get('cores')}, return code {writer.get('returncode')}")
print("the file was last modified:    ", UTCDateTime(dtct_file.stat().st_mtime))
print("\nlast lines of the run log (start, command, stages, cores, return code):")
for line in RUNS_LOG.read_text(encoding="utf-8").splitlines()[-4:]:
    run = json.loads(line)
    print(" ", run["ts"], run["command"], run.get("stage_from", run.get("branch")), "->", run.get("through", ""),
          "cores", run["cores"], "return code", run["returncode"])
print("\nnewest files in PocketQuake's log folder:", sorted(p.name for p in (RUN_DIR / "sgtl_logs").iterdir())[-4:])

The two bootstraps side by side (95 % half-widths in m), and HypoDD's dt.cc relocation against the HYPOINVERSE
locations:

In [ ]:
boots = {seed: pd.read_csv(path, comment="#").set_index("id") for seed, path in boot_copies.items()}
side = pd.concat({f"seed {seed}": table[["n_boot", "ex95", "ey95", "ez95"]] for seed, table in boots.items()},
                 axis=1)
print(side.round(1).to_string())
ratio = boots[BOOT_SEED_CHANGED][["ex95", "ey95", "ez95"]] / boots[BOOT_SEED][["ex95", "ey95", "ez95"]]
print(f"\nratio of the half-widths (seed {BOOT_SEED_CHANGED} / seed {BOOT_SEED}): from "
      f"{np.nanmin(ratio.to_numpy()):.2f} to {np.nanmax(ratio.to_numpy()):.2f}")
offset = 1000.0 * (sets["HypoDD dt.cc"].mean(axis=0) - start_km.mean(axis=0))
print(f"HypoDD dt.cc centroid minus HYPOINVERSE centroid (east, north, down; m): {np.round(offset, 0)}")
same = sorted(dd_runs["first"].values()) == sorted(dd_runs["rerun"].values())
print("the unchanged rerun wrote the same bytes (SHA-256 of every copy):", same)

**Relative precision and a common shift answer different questions.** The left panel shows your bootstrap's
95% half-widths after replicas have been aligned. The right shows the centroid displacement between two
methods, not an error measured against a known true location. The scales differ deliberately: the bootstrap
does not measure a common translation of the whole cluster or common velocity-model errors.

**Look:** could a very tight relative cluster still have uncertain absolute depth? Which additional
observations or model tests would constrain the common position that the aligned bootstrap leaves out?

In [ ]:
from IPython.display import Image, display

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
visual_boot = boots[BOOT_SEED]
for column, name in (("ex95", "East"), ("ey95", "North"), ("ez95", "Depth")):
    axes[0].plot(visual_boot.index.astype(str), visual_boot[column], "o-", ms=4, label=name)
axes[0].set(xlabel="Event ID", ylabel="Aligned bootstrap 95% half-width (m)", title="Relative position uncertainty")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)
axes[1].bar(["East", "North", "Down"], offset, color=["tab:blue", "tab:orange", "0.5"])
axes[1].axhline(0, color="0.3", lw=0.7)
axes[1].set(ylabel="HypoDD minus HYPOINVERSE centroid (m)", title="Difference between methods, not known error")
fig.savefig(Path(FIGURE_DIR) / "M41_relative_and_common_position.png", dpi=150)
display(Image(filename=str(Path(FIGURE_DIR) / "M41_relative_and_common_position.png")))
plt.close(fig)

**Explain (3-5 sentences):** Was the colleague's file written by their run: what did the run log, the file's
modification time and the log folder say, and what had happened to the run (look at the server rules of `st.pq`)?
Was the unchanged rerun identical, and why does that matter before any comparison? How much did the half-widths
change with the second seed, and what does that say about 200 replicas? Compare the half-widths with the offset of
HypoDD's centroid from HYPOINVERSE's: which uncertainty does the bootstrap leave out?

## Wrap-up

- `results/hypodd/`: your copies of the dt.cc products (`first/`, `rerun/`) and of the two bootstrap tables;
  `work/.sgtl/runs.jsonl` logs every run with the fingerprints of its products.
- `results/M41_cc_times.csv`: your cross-correlation differential times next to PocketQuake's.
- `figures/correlation_function.png`, `figures/cc_times_vs_pocketquake.png` and `figures/relocations.png`.

The waveforms of this module are raw counts: a correlation coefficient does not depend on the gain, so no
instrument response was needed (the course's station metadata, StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong
National University, reference key `ahn2024`, served modules M03 and M10). The KMA catalog is used under the Korea
Open Government License Type 1 (attribution).

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M41")

### Going further (optional, not checked)

1. Rerun the bootstrap with seed 1 instead of `BOOT_SEED_CHANGED`. Why do the half-widths hardly change? (Read the
   docstring lines printed in Part 7.)
2. Relocate with your functions from the KMA catalog locations instead of HYPOINVERSE's. Does the shape change?
   Does the centroid?
3. Add your cross-correlation times of Part 6 with a coefficient of at least 0.8 to your own relocation (with a
   weight of 1 for P and 0.8 for S, as HypoDD's later weighting sets do). How much does the shape change?
4. In the cell that sets the station limits of Part 6, take the crossover distances of the shallowest event
   instead of the deepest one above the boundary. How many stations come in, and how do the RMS and the shape
   differences from HypoDD change? For which events is a head wave the first arrival at the stations that came in?

Next: module M50 turns to the source itself, with the focal sphere and nodal planes; M51 finds mechanisms of the
Sangju earthquakes from PhaseNet+'s first motions.